# logit-thermo A2-20 第 2 段：7B 級 2 本（Qwen2.5-7B・Mistral-7B-v0.3）に、予言した罰 h* をかける——ランタイム 3/3（GPU、A100）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。
- **ランタイム：A100（40GB 以上）**。ハイメモリにしなくて大丈夫です（CPU のメモリが 48GB 以上あれば動きます。足りなければ ⚠ を送ってランタイムを切ります）。A100 でなければ、⚠ を送ってランタイムを切ります。
- **3 本のノートブック（ランタイム 1/3・2/3・3/3）を、別々のランタイムで同時に回してください。** 仕事は 12 個（モデル 2 本 × 腕 2 つ（罰なしの対照・予言した罰 h*）× 記事の 3 つの組）で、3 つのランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行きます。1 つの仕事は 25〜45 分くらい、3 つ同時なら 2〜3 時間の見込みです（CU は 3 つ合わせて 30〜40 くらい）。
- 作り：A2-19 の新しい BBC の記事 120 本（目録で照らし合わせます）、サンプル 4、512 歩、q つき。h* と温度は第 1 段から予言して封をしたもの（`results/A2_20/hstar7b.json`、ハッシュで確かめます）：Qwen2.5-7B は h* ＝ −0.675・温度 0.30〜0.60、Mistral-7B は h* ＝ −0.345・温度 0.19〜0.49。モデルは fp32、TF32 なし。
- **データは、第 2 段が終わるまで開けません。** 結果も記事の中身も表示しません。
- 計算のコード：これまでのデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_18_collect.py`（`0b9d203ebe0a71e6…`）・`a2_19_collect.py`（`de4d86520b5ab6a4…`）・`a2_20_collect.py`（`29d5dbd7a0562043…`）は、Drive の `A2_explore/` の .py から読んでハッシュで確かめる。
- 保存先 `A2_20_s2/`。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします。
- **通知**：🚀・仕事ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-20 第2段 7B に予言した罰（Qwen2.5-7B・Mistral-7B、対照・h*、記事 120 本、ランタイム 3/3、A100）」。終わったら、またはどのセルでもエラーで止まったら、ランタイムを切ります（手で止めたときも切れます）。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-20 第2段 7B に予言した罰（Qwen2.5-7B・Mistral-7B、対照・h*、記事 120 本、ランタイム 3/3、A100）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-20 stage2 7B predicted penalty worker 3of3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch, psutil
_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
_gmem = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
_ram = psutil.virtual_memory().total
if 'A100' not in _gpu or _gmem < 39e9 or _ram < 48e9:
    notify(f'⚠️ ランタイムが合いません（GPU {_gpu}、GPU のメモリ {_gmem / 1e9:.0f}GB、CPU のメモリ {_ram / 1e9:.0f}GB）。A100（40GB 以上）を選び、CPU のメモリが 48GB 以上あることを確かめてください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
def _cut_on_error(result):
    # どのセルでエラーになっても（A2-19 ではセル 2 で止まったときに切れなかった）、⚠ を送ったあとでランタイムを切る。手で止めたときも切れます
    if result.error_in_exec is not None and globals().get('AUTO_DISCONNECT', True):
        time.sleep(10)
        try:
            from google.colab import drive as _d; _d.flush_and_unmount()
        except Exception:
            pass
        from google.colab import runtime as _r; _r.unassign()
if not globals().get('_CUT_ON'):
    get_ipython().events.register('post_run_cell', _cut_on_error); _CUT_ON = True

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', 'SRCG': '49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCD2, SRCG = _s['SRCD2'], _s['SRCG']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
def take_file(name, want):
    """Drive の logit-thermo のフォルダから、名前とハッシュが合う .py のファイルを読む。"""
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/' + name, recursive=True)):
        s = open(p, encoding='utf-8').read()
        if hashlib.sha256(s.encode()).hexdigest() == want:
            return s
    raise AssertionError(('Drive に ' + name + ' （ハッシュが合うもの）が見つかりません', want))
SRC17 = take_file('a2_18_collect.py', '0b9d203ebe0a71e67d3c419897212c2091f04697c0187ac9475f9112eeea6496')
SRC19 = take_file('a2_19_collect.py', 'de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef')
SRC20 = take_file('a2_20_collect.py', '29d5dbd7a05620433f5bf0d1ad9dc3c9e130f89d8ed66622139149d51fea99d6')
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
exec(SRC17)
exec(SRC19)
exec(SRC20)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2・A2-19 と同じか目録で確かめる）→ 封をした予言（ハッシュで確かめる）→ 仕事を取りに行って回す → 保存（結果も記事も表示しない）
import json, sys, transformers
HSTAR_JSON = '{\n "models": {\n  "qwen2.5-7b": {\n   "temps1": [\n    0.2,\n    0.25,\n    0.3,\n    0.35,\n    0.4,\n    0.45,\n    0.5,\n    0.55,\n    0.6,\n    0.65,\n    0.7,\n    0.75,\n    0.8,\n    0.85,\n    0.9\n   ],\n   "fit_all": {\n    "ok": true,\n    "Tm": 0.4978789519748467,\n    "use_idx": [\n     0,\n     1,\n     2,\n     3,\n     4,\n     5,\n     6,\n     7,\n     8,\n     9,\n     10,\n     11,\n     12,\n     13,\n     14\n    ]\n   },\n   "T_all": 0.4978789519748467,\n   "window_first": [\n    0.35,\n    0.4,\n    0.45,\n    0.5,\n    0.55,\n    0.6,\n    0.65\n   ],\n   "window": [\n    0.4,\n    0.45,\n    0.5,\n    0.55,\n    0.6,\n    0.65,\n    0.7\n   ],\n   "Tm0": 0.5530119526201538,\n   "a": -3.980065859205133,\n   "b": 2.201023992355841,\n   "dH_cal": 0.8342871590828589,\n   "dm": 0.7505232890506903,\n   "g_T": [\n    0.5859568160409226,\n    2.167281045957789,\n    1.2936825471819888,\n    0.9681849291817286,\n    -0.2089140584581506,\n    1.876665533491467,\n    2.429841309352332\n   ],\n   "gbar": 1.3018140175354398,\n   "s_clapeyron": 0.49748859855530986,\n   "use_idx": [\n    0,\n    1,\n    2,\n    3,\n    4,\n    5,\n    6\n   ],\n   "target": {\n    "dT": -0.1,\n    "Tstar": 0.4530119526201538,\n    "hstar": -0.674887918993807,\n    "grid": [\n     0.3,\n     0.35,\n     0.4,\n     0.45,\n     0.5,\n     0.55,\n     0.6\n    ],\n    "r_clapeyron": 3.3574904500213854\n   },\n   "name": "qwen2.5-7b",\n   "ok": true,\n   "why": null\n  },\n  "mistral-7b-v0.3": {\n   "temps1": [\n    0.2,\n    0.25,\n    0.3,\n    0.35,\n    0.4,\n    0.45,\n    0.5,\n    0.55,\n    0.6,\n    0.65,\n    0.7,\n    0.75,\n    0.8,\n    0.85,\n    0.9\n   ],\n   "fit_all": {\n    "ok": true,\n    "Tm": 0.38635426661955946,\n    "use_idx": [\n     0,\n     1,\n     2,\n     3,\n     4,\n     5,\n     6,\n     7,\n     8,\n     9,\n     10,\n     11,\n     12,\n     13,\n     14\n    ]\n   },\n   "T_all": 0.38635426661955946,\n   "window_first": [\n    0.25,\n    0.3,\n    0.35,\n    0.4,\n    0.45,\n    0.5,\n    0.55\n   ],\n   "window": [\n    0.3,\n    0.35,\n    0.4,\n    0.45,\n    0.5,\n    0.55,\n    0.6\n   ],\n   "Tm0": 0.44188708116324366,\n   "a": -3.2543361744689676,\n   "b": 1.4380491132600486,\n   "dH_cal": 0.659422533766943,\n   "dm": 0.7186119950672035,\n   "g_T": [\n    3.068407507353744,\n    2.5488401101195812,\n    3.3949799817463053,\n    1.8504471019333755,\n    2.8402137635397806,\n    2.9818239541199363,\n    2.6052832859312116\n   ],\n   "gbar": 2.755713672106276,\n   "s_clapeyron": 0.4815506609626576,\n   "use_idx": [\n    0,\n    1,\n    2,\n    3,\n    4,\n    5,\n    6\n   ],\n   "target": {\n    "dT": -0.1,\n    "Tstar": 0.3418870811632436,\n    "hstar": -0.3454185305725316,\n    "grid": [\n     0.19,\n     0.24,\n     0.29,\n     0.34,\n     0.39,\n     0.44,\n     0.49\n    ],\n    "r_clapeyron": 1.6633652170595252\n   },\n   "name": "mistral-7b-v0.3",\n   "ok": true,\n   "why": null\n  }\n },\n "sha256": {\n  "scripts/a2_20_predict.py": "89bd57664359aaf1ce50f287f13f39bd524d37bf4bdd16c350429a98331d0ab3",\n  "scripts/a2_20_analyze.py": "16bb40e8a0a2f26460532b32cf2f52b1d586b60e5dd2689c8310c5bc41f77359",\n  "scripts/a2_20_collect.py": "29d5dbd7a05620433f5bf0d1ad9dc3c9e130f89d8ed66622139149d51fea99d6",\n  "scripts/a2_19_analyze.py": "66db0e8bf0aa6a437b9251b3f66c797c1ee304136f9daeda85191b95666b2265",\n  "scripts/a2_18_analyze.py": "5b55c3df8e84c2104ec5ea1a83f48cb13f149cd1b280a97832a413b4b79cdd0e",\n  "scripts/a2_18_predict.py": "72b2ed5d617b54241088289fbc67444862f6d7bca679a5871ef18a24ce34f681",\n  "scripts/a2_16_analyze.py": "7c5a60c2f8077d2d0e4051a65ef5b075ac24517d34a5069aa51271a0b73faf0f"\n }\n}'
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_gpre(), self_test_a18(), self_test_a19(), self_test_a20())
    assert hashlib.sha256(HSTAR_JSON.encode()).hexdigest() == '18e87a54eec46f7dd8397af476a3dc834b39386f3f06a25f759f817a03f6c729', '封をした予言と違います'
    open('/content/hstar7b.json', 'w', encoding='utf-8').write(HSTAR_JSON)
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_20_s2', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    check_docs20(meta, json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json')), 'docs')
    GROUPS, META19 = load_docs19('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json')
    check_docs20(META19, json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_19/docs_meta_w0.json')), 'groups')
    JOBS = a20_jobs2(['/content/hstar7b.json'])
    assert len(JOBS) == 12 and sorted({j[0] for j in JOBS}) == ['mistral-7b-v0.3', 'qwen2.5-7b'], JOBS
    N_DONE, LEFT = run_a20(JOBS, 2, 3, lambda j: GROUPS[j[4]], '/content/drive/MyDrive/logit-thermo/A2_20_s2', 'cuda', 20, notify=lambda s: notify(s, tags='warning') if s.startswith('⚠') else notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(worker=2, done=N_DONE, left=LEFT, samples=GP_S, chunk=20, hstar_sha256='18e87a54eec46f7dd8397af476a3dc834b39386f3f06a25f759f817a03f6c729',
                   a20_sha256='29d5dbd7a05620433f5bf0d1ad9dc3c9e130f89d8ed66622139149d51fea99d6', a19_sha256='de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef', a18_sha256='0b9d203ebe0a71e67d3c419897212c2091f04697c0187ac9475f9112eeea6496', dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_20_s2' + '/run_meta_worker2.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-20 第2段 7B に予言した罰（ランタイム 3/3）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：A2-20 第2段 7B に予言した罰（ランタイム 3/3）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
